In [20]:
import pandas as pd
df =pd.read_excel("./1.xlsx",encoding='gbk')

In [21]:
#keywords=['26283121778226','26076662786226','26067223087226','25779902720226','26066979183226','25785921087226','26066692527226','26067006475226']
 #10000229167469,10000229407532,10000265046381,10000270409428,10000269553640,10000273809516
#df['商品ID'].isin(keywords)
#df['商品ID'] = df['商品ID'].astype(str)


#删除其他行
#df.drop(index= df[ df['商品ID'].isin(keywords)==False].index,inplace=True)
#df.reset_index(drop=True,inplace= True)

In [22]:
df["实付款"] = df["实付款"].astype(str).str.replace("¥", "").astype(float)

In [23]:
from datetime import datetime
df["订单创建时间"] = pd.to_datetime(df["订单创建时间"],format = "%Y-%m-%d %H:%M:%S")
df["订单创建时间"] = df["订单创建时间"].map(lambda x: x.strftime('%Y-%m-%d'))
df1 = pd.read_excel("商品编码.xlsx")
df1.rename(columns={"商品名":"产品名称","商家编码":"SKU编码"},inplace=True)
df1 = df1[["SKU编码","产品名称"]]
df= pd.merge(df,df1,on="SKU编码",how="left")

In [24]:
# 缺码表
lackCode = df[df['产品名称'].isnull()]['SKU编码'].value_counts()
lackCode

Series([], Name: SKU编码, dtype: int64)

In [25]:
lackCode.to_excel('缺码表.xlsx')

In [26]:
df['订单数'] = df['成交数量']

In [27]:
df["下单单量"] = df['订单数']

In [28]:
# 清除未支付订单
from datetime import datetime
df["订单支付时间"] = df["订单支付时间"].fillna(pd.to_datetime('1970-01-01'))
df["订单支付时间"] = pd.to_datetime(df["订单支付时间"],format = "%Y-%m-%d %H:%M:%S")
df["订单支付时间"] = df["订单支付时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [29]:
df["发货时间"] = df["发货时间"].fillna(pd.to_datetime('1970-01-01'))
df["发货时间"] = pd.to_datetime(df["发货时间"],format = "%Y-%m-%d %H:%M:%S")
df["发货时间"] = df["发货时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [30]:
df["支付单量"] = 0
df["发货单量"] = 0
df["支付金额"] = 0
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01":
        df.loc[i,"发货单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单支付时间"] != "1970-01-01":
        df.loc[i,"支付单量"] = df.loc[i,'订单数']
        df.loc[i,"支付金额"] = df.loc[i,'实付款']

In [31]:
df["售后状态"] = df["售后状态"].astype(str)

In [32]:
df["商品ID"] = df["商品ID"].astype(str)

In [33]:
df["售后状态"].value_counts()

nan      85
退款成功     23
待卖家收货     9
退款关闭      2
待卖家处理     1
待买家退货     1
Name: 售后状态, dtype: int64

In [34]:
df["商品ID"].value_counts()

26088887665676    91
26088849646676    22
26837856692676     3
26100751032676     1
26220643042676     1
25964385307676     1
26837895558676     1
26185748880676     1
Name: 商品ID, dtype: int64

In [35]:
df['物流信息'].value_counts()

已签收      67
未知       21
商家已接单    10
运输中       8
待取件       7
派送中       3
已退签       2
已发货       2
物流异常      1
Name: 物流信息, dtype: int64

In [36]:
for i in range (0,len(df)):
    if df.loc[i,"订单状态"] == "已收货" and (df.loc[i,'售后状态'] == 'nan' or df.loc[i,'售后状态'] == '退款关闭'):
        df.loc[i,"订单状态"] = "交易成功"

In [37]:
df["已签收"] = 0
df["发货中"] = 0
df["售后中"] = 0
df["已退款"] = 0
df["预计签收数量"] = 0
df["发货前退款单量"] = 0
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "交易成功" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"已签收"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "已发货" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"发货中"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"售后状态"] == "待买家处理" or df.loc[i,"售后状态"] == "待买家退货" or df.loc[i,"售后状态"] == "待卖家处理" or df.loc[i,"售后状态"]=="待卖家收货" or df.loc[i,"售后状态"]=="待平台判定":
        df.loc[i,"售后中"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01" and df.loc[i,"售后状态"] == "退款成功":
        df.loc[i,"已退款"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单支付时间"] != "1970-01-01"  and  df.loc[i,"发货时间"] == "1970-01-01" and df.loc[i,"售后状态"] == "退款成功":
        df.loc[i,"发货前退款单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"预计签收单量"] = df.loc[i,"订单数"]

In [38]:
import numpy as np
result = pd.pivot_table(df,index=['订单创建时间',"产品名称"] , values = ['下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额'] , aggfunc = np.sum)
result = result.reset_index()
result = result[['订单创建时间',"产品名称",'下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额']]

In [39]:
result.to_excel("111.xlsx",index=False)

In [40]:
# 拼接店铺消耗表
df_consume = pd.read_excel("快手店铺消耗8号.xlsx")
df_consume = df_consume[['日期','产品','花费','转化数']]
temp = df_consume.日期.str.extract('(\d+).*?(\d+).*?(\d+)') 
df_consume['订单创建时间'] = temp[0] + "-" + temp[1] + '-' + temp[2]
df_consume.rename(columns={"产品":"产品名称"},inplace=True)
del df_consume['日期']
df_consume = pd.pivot_table(df_consume,index=['订单创建时间','产品名称'] , values = ['花费','转化数'] , aggfunc = np.sum)
df_consume = df_consume.reset_index()
df_consume["订单创建时间"] = pd.to_datetime(df_consume["订单创建时间"],format = "%Y-%m-%d")
df_consume["订单创建时间"] = df_consume["订单创建时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [41]:
final = pd.merge(result,df_consume,on = ['订单创建时间','产品名称'],how = 'left')
final = final[['订单创建时间',"产品名称",'转化数','下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额']]
final.rename(columns={"转化数":"总广告单量"},inplace=True)

In [42]:
final

,订单创建时间,产品名称,总广告单量,下单单量,支付单量,发货单量,已签收,发货中,售后中,已退款,发货前退款单量,预计签收单量,支付金额
0,2026-10-13,16888运动背心,3.0,5,5,5,5,0,0,0,0,5.0,349.5
1,2026-10-16,16888运动背心,0.0,2,1,1,1,0,0,0,0,1.0,69.9
2,2026-10-18,16888运动背心,1.0,1,1,1,1,0,0,0,0,1.0,69.9
3,2026-10-19,16888运动背心,1.0,1,1,1,1,0,0,0,0,1.0,69.9
4,2026-10-20,16888运动背心,2.0,1,1,0,0,0,0,0,1,0.0,39.9
5,2026-10-21,16888运动背心,7.0,9,7,7,7,0,0,0,0,7.0,489.3
6,2026-10-22,16888运动背心,4.0,5,4,3,1,0,0,2,1,1.0,279.6
7,2026-10-23,16888运动背心,4.0,6,5,3,2,0,0,1,2,2.0,349.5
8,2026-10-24,16888运动背心,5.0,5,5,4,1,0,1,2,1,1.0,349.5
9,2026-10-25,16888运动背心,9.0,11,10,9,3,3,2,1,1,6.0,699.0


In [43]:
final.to_excel('直播签收汇总.xlsx')